**Part 3 · LangGraph** · step 18 of 25  
[← 17 State & reducers](17_state_and_reducers.ipynb) · [🏠 Roadmap](../README.md) · [19 Research chatbot →](19_research_chatbot_multiple_tools.ipynb)

# 18 · Tool-Calling Agent as a Graph (ReAct loop)

Here we build by hand what `create_agent` (notebook 10) gives you for free:

```
START → llm_agent ──tools_condition──► tools ─┐
           ▲                                  │
           └──────────────────────────────────┘
           └──(no tool calls)──► END
```

| Prebuilt | Purpose |
|---|---|
| `ToolNode(tools)` | Executes every tool call in the last `AIMessage` and returns `ToolMessage`s |
| `tools_condition` | Router: returns `"tools"` if the last message has tool calls, else `END` |

In [ ]:
from dotenv import load_dotenv

# Reads API keys (OPENAI_API_KEY, GROQ_API_KEY, GOOGLE_API_KEY, TAVILY_API_KEY)
# from the .env file at the AgenticAI root. load_dotenv() walks up parent
# folders until it finds one, so every notebook can share a single .env.
load_dotenv()

In [ ]:
from typing import Annotated
from typing_extensions import TypedDict

from langchain.chat_models import init_chat_model
from langchain.messages import AnyMessage, HumanMessage
from langchain.tools import tool
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition

## Tools and an LLM that knows about them

In [ ]:
@tool
def add_two_number(a: int, b: int) -> int:
    """Add two numbers and return the result."""
    return a + b


@tool
def subtract_two_number(a: int, b: int) -> int:
    """Subtract b from a and return the result."""
    return a - b


@tool
def multiply_two_number(a: int, b: int) -> int:
    """Multiply two numbers and return the result."""
    return a * b


input_tools = [add_two_number, subtract_two_number, multiply_two_number]
llm_with_tools = init_chat_model("gpt-4o").bind_tools(input_tools)

## `bind_tools` does NOT run tools — it only asks
`bind_tools` just sends the tool **names, descriptions and argument schemas** to the model.
When the model wants a tool, it returns an `AIMessage` whose `tool_calls` is a *request*
("please call `add_two_number` with `{a: 2, b: 2}`"). Nothing is executed — `content` is usually empty
because the model is waiting for the result.

In [ ]:
ai_msg = llm_with_tools.invoke([HumanMessage("add 2 + 2")])
print("tool_calls:", ai_msg.tool_calls)
print("content   :", repr(ai_msg.content))

### Running the tool by hand
**You** must execute the tool and send the result back as a `ToolMessage` (linked by `tool_call_id`).
Passing the whole tool-call dict to `tool.invoke(...)` returns a ready-made `ToolMessage`.

Flow: **model asks → you run the tool → you send back the result → model answers.**
Repeating this until the model stops asking for tools *is* the agent loop.

In [ ]:
tools_by_name = {t.name: t for t in input_tools}

messages = [HumanMessage("add 2 + 2"), ai_msg]
for tc in ai_msg.tool_calls:
    tool_msg = tools_by_name[tc["name"]].invoke(tc)  # -> ToolMessage(tool_call_id=tc["id"])
    messages.append(tool_msg)

final = llm_with_tools.invoke(messages)
final.content

## `ToolNode` — the node that actually runs tools
In LangGraph the work is split:

| Piece | Job |
|---|---|
| LLM node (`bind_tools`) | **decides** which tool to call → `AIMessage.tool_calls` |
| `ToolNode(tools)` | **executes** every tool call in the last `AIMessage` (in parallel) → `ToolMessage`s appended to `messages` |
| `tools_condition` | **routes**: tool calls present → `"tools"`, otherwise → `END` |

Gotchas:
- Give `ToolNode` the **same tools list** you passed to `bind_tools`, or the model may request a tool `ToolNode` doesn't have.
- `tools_condition` routes to a node literally named `"tools"` — use that name (or write your own router).
- If a tool raises, `ToolNode` by default returns the error text as a `ToolMessage` so the model can retry
  instead of crashing the graph (configurable via `handle_tool_errors`).

`ToolNode` is just a runnable — you can call it on its own with the `ai_msg` from above:

In [ ]:
ToolNode(input_tools).invoke({"messages": [ai_msg]})

## Build the graph

In [ ]:
class State(TypedDict):
    """Chat history; add_messages appends new messages."""

    messages: Annotated[list[AnyMessage], add_messages]


def llm_agent(state: State) -> dict:
    """Call the LLM with the full history and return ONLY its new message."""
    return {"messages": [llm_with_tools.invoke(state["messages"])]}


builder = StateGraph(State)
builder.add_node("llm_agent", llm_agent)
builder.add_node("tools", ToolNode(input_tools))

builder.add_edge(START, "llm_agent")
builder.add_conditional_edges("llm_agent", tools_condition)  # -> "tools" or END
builder.add_edge("tools", "llm_agent")                       # loop back after tools run

graph = builder.compile()

In [ ]:
from IPython.display import Image, display


def show_graph(compiled_graph):
    """Render a compiled graph as a PNG.

    draw_mermaid_png() uses the mermaid.ink web service; if you are offline
    it falls back to printing the Mermaid source (paste it into mermaid.live).
    """
    try:
        display(Image(compiled_graph.get_graph().draw_mermaid_png()))
    except Exception:
        print(compiled_graph.get_graph().draw_mermaid())


show_graph(graph)

In [ ]:
result = graph.invoke({"messages": [HumanMessage("add 2 + 2 and 2 * 2 and 2+8*2+2")]})
for m in result["messages"]:
    m.pretty_print()

### ⚠️ Mistakes made (and fixed) while learning this
1. **Extra `add_edge("llm_agent", END)`** next to `add_conditional_edges("llm_agent", tools_condition)`.
   `tools_condition` already routes to `END`; a second static edge makes *both* paths fire.
2. **Returning the whole list** (`state["messages"] + [reply]`) — with `add_messages` return only the new message.
3. **Reading a key that may be missing** (`state["count"]`) — use `state.get("count", 0)`.

## Extra state keys — a call counter
State can hold more than messages. Keys without a reducer are overwritten by each update.

In [ ]:
class CounterState(TypedDict):
    """Messages plus how many times the LLM node ran."""

    count: int
    messages: Annotated[list[AnyMessage], add_messages]


def counting_llm(state: CounterState) -> dict:
    """Call the LLM and increment the counter."""
    return {
        "count": state.get("count", 0) + 1,
        "messages": [llm_with_tools.invoke(state["messages"])],
    }


workflow = StateGraph(CounterState)
workflow.add_node("llm_agent", counting_llm)
workflow.add_node("tools", ToolNode(input_tools))
workflow.add_edge(START, "llm_agent")
workflow.add_conditional_edges("llm_agent", tools_condition)
workflow.add_edge("tools", "llm_agent")
counter_graph = workflow.compile()

resp = counter_graph.invoke({"count": 0, "messages": [HumanMessage("add 2 + 2, then multiply the result by 10")]})
print("LLM calls:", resp["count"])
resp["messages"][-1].content

## Memory — why "what is my name?" failed
Each `invoke` without a checkpointer starts from an empty state, so the graph forgets everything.
Compile with a **checkpointer** and pass a `thread_id` (same idea as agents in notebook 11).

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver

memory_graph = workflow.compile(checkpointer=InMemorySaver())
config = {"configurable": {"thread_id": "ujjawal-1"}}

memory_graph.invoke({"messages": [HumanMessage("my name is Ujjawal Kant and I like coffee")]}, config)
resp = memory_graph.invoke({"messages": [HumanMessage("what is my name?")]}, config)
resp["messages"][-1].content

## Streaming the graph: `stream_mode`
`invoke` waits until the graph reaches `END`. `stream` yields **while** the graph runs. It's the same
API you used on agents in 15b, because `create_agent` *is* a compiled LangGraph graph. The difference
here is that **you** chose the node names (`llm_agent`, `tools`), so those are what show up in the chunks.

| `stream_mode` | Each item is… | Use it for |
|---|---|---|
| `"values"` | the **full state** after each step | debugging, simple UIs |
| `"updates"` | `{node_name: what_that_node_returned}` | progress logs ("llm_agent → tools → llm_agent") |
| `"messages"` | `(message_chunk, metadata)`: LLM **tokens** from inside any node | chat UIs (typing effect) |
| `"custom"` | whatever a node sends via `get_stream_writer()` | your own status events |
| `"debug"` | everything (tasks, checkpoints, results) | deep debugging only |

Default for `graph.stream(...)` is `"updates"`.

### 1. `"values"`: full state after every super-step

In [ ]:
question = {"messages": [HumanMessage("add 2 + 2, then multiply the result by 10")]}

for state in graph.stream(question, stream_mode="values"):
    state["messages"][-1].pretty_print()

### 2. `"updates"`: only what each node returned
The key is the node name you passed to `add_node`. This is how you see the ReAct loop happen step by step.

In [ ]:
for chunk in graph.stream(question, stream_mode="updates"):
    for node, update in chunk.items():
        last = update["messages"][-1]
        if node == "llm_agent" and last.tool_calls:
            print(f"[{node}] wants: {[(c['name'], c['args']) for c in last.tool_calls]}")
        elif node == "llm_agent":
            print(f"[{node}] final answer: {last.content}")
        else:
            for m in update["messages"]:
                print(f"[{node}] {m.name} -> {m.content}")

### 3. `"messages"`: token-by-token, even though the node calls `invoke`
`llm_agent` uses `llm_with_tools.invoke(...)`, not `.stream(...)`. LangGraph still streams its tokens:
when you ask for `"messages"`, it hooks into every chat model call made inside a node.
`metadata["langgraph_node"]` tells you which node the token came from.

In [ ]:
for token, metadata in graph.stream(question, stream_mode="messages"):
    if metadata["langgraph_node"] == "llm_agent" and token.content:
        print(token.content, end="", flush=True)

### 4. `"custom"`: send your own events from a node with `get_stream_writer()`
In 15b tools used `runtime.stream_writer`. In a plain LangGraph node you grab the writer with
`get_stream_writer()` (from `langgraph.config`). Anything you pass to it arrives in the `"custom"` stream;
it's ignored when you call `invoke` or stream other modes.

In [ ]:
from langgraph.config import get_stream_writer


def llm_agent_with_status(state: State) -> dict:
    """Same LLM node, but it reports progress through the custom stream."""
    writer = get_stream_writer()
    writer({"status": f"thinking over {len(state['messages'])} messages..."})
    reply = llm_with_tools.invoke(state["messages"])
    writer({"status": "calling tools" if reply.tool_calls else "done"})
    return {"messages": [reply]}


status_builder = StateGraph(State)
status_builder.add_node("llm_agent", llm_agent_with_status)
status_builder.add_node("tools", ToolNode(input_tools))
status_builder.add_edge(START, "llm_agent")
status_builder.add_conditional_edges("llm_agent", tools_condition)
status_builder.add_edge("tools", "llm_agent")
status_graph = status_builder.compile()

for event in status_graph.stream(question, stream_mode="custom"):
    print(event)

### 5. Several modes at once, with memory
Pass a list and each item becomes a `(mode, chunk)` tuple. A checkpointed graph streams the same way;
just pass the `config` with the `thread_id`. Use `astream` (async) inside a web server, like in 15b.

In [ ]:
stream_graph = status_builder.compile(checkpointer=InMemorySaver())
stream_config = {"configurable": {"thread_id": "stream-1"}}

for mode, chunk in stream_graph.stream(question, stream_config, stream_mode=["messages", "custom"]):
    if mode == "custom":
        print(f"\n   [status] {chunk['status']}")
    else:
        token, metadata = chunk
        if metadata["langgraph_node"] == "llm_agent" and token.content:
            print(token.content, end="", flush=True)

In [ ]:
# Async twin: same arguments, `async for` + `astream`. Jupyter can `await` at top level.
async for token, metadata in stream_graph.astream(
    {"messages": [HumanMessage("now subtract 7 from that")]}, stream_config, stream_mode="messages"
):
    if metadata["langgraph_node"] == "llm_agent" and token.content:
        print(token.content, end="", flush=True)

## Revision notes
- `ToolNode` executes tool calls, `tools_condition` routes to `"tools"` or `END`, and `add_edge("tools", "llm_agent")` closes the loop.
- With `add_messages`, a node returns only its **new** messages.
- Memory = `compile(checkpointer=...)` + `{"configurable": {"thread_id": ...}}`.
- `graph.stream(input, config, stream_mode=...)`: `values` = full state · `updates` = `{node: diff}` (default) ·
  `messages` = `(token, metadata)` from any LLM call inside a node · `custom` = `get_stream_writer()` events.
- A list of modes yields `(mode, chunk)`. `astream` is the async version.

---
✅ **Done with 18 Tool-calling agent graph.** Next: [19 Research chatbot →](19_research_chatbot_multiple_tools.ipynb)  
[🏠 Roadmap](../README.md)